# 02 — ETL: criação dos marts

Transforma o dataset bruto em três tabelas agregadas (marts) salvas como CSV. Esses arquivos serão a **única fonte de dados** do Streamlit.

| Mart | Granularidade | Uso no app |
|---|---|---|
| `mart_ads_performance.csv` | Um anúncio por linha | What-if, dispersão, ROI por campanha |
| `mart_aggregated.csv` | Um registro por campanha | Visão geral, gráficos de barra |
| `mart_roi_parameters.csv` | Parâmetros globais | Valor padrão de `avg_order_value` e taxa de conversão |

**Entrada:** `data/raw/conversion_data.csv`  
**Saída:** `data/processed/*.csv`

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

In [10]:
RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# Parâmetro de negócio: valor médio por pedido (ajustável pelo usuário no Streamlit)
AVG_ORDER_VALUE = 50.0

## 1. Leitura e limpeza base

In [11]:
df = pd.read_csv(RAW / "conversion_data.csv")

# Remove anúncios com gasto zero (irrelevantes para o modelo de ROI)
df = df[df["spent"] > 0].copy()

# Métricas calculadas por anúncio
df["revenue"]         = df["approved_conversion"] * AVG_ORDER_VALUE
df["roi"]             = (df["revenue"] - df["spent"]) / df["spent"]
df["conversion_rate"] = df["approved_conversion"] / df["clicks"].replace(0, np.nan)
df["cpa"]             = df["spent"] / df["approved_conversion"].replace(0, np.nan)

print(f"Registros após limpeza: {len(df)}")

Registros após limpeza: 936


## 2. Mart 1 — Desempenho por anúncio

Granularidade original (um anúncio por linha). Usado nas abas What-If e Sensibilidade.

In [12]:
colunas_ads = [
    "ad_id", "xyz_campaign_id", "age", "gender", "interest",
    "impressions", "clicks", "spent",
    "total_conversion", "approved_conversion",
    "revenue", "roi", "conversion_rate", "cpa"
]

mart_ads = df[colunas_ads].reset_index(drop=True)

out_ads = PROCESSED / "mart_ads_performance.csv"
mart_ads.to_csv(out_ads, index=False)

print(f"mart_ads_performance → {len(mart_ads)} linhas | {out_ads}")
mart_ads.head(3)

mart_ads_performance → 936 linhas | ..\data\processed\mart_ads_performance.csv


,ad_id,xyz_campaign_id,age,gender,interest,impressions,clicks,spent,total_conversion,approved_conversion,revenue,roi,conversion_rate,cpa
0,708746,916,30-34,M,15,7350,1,1.43,2,1,50.0,33.965036,1.0,1.43
1,708749,916,30-34,M,16,17861,2,1.82,2,0,0.0,-1.000000,0.0,NaN
2,708815,916,30-34,M,28,4259,1,1.25,1,0,0.0,-1.000000,0.0,NaN


## 3. Mart 2 — Agregação por campanha

Consolida ao nível de campanha. Usado na aba Visão Geral.

In [13]:
mart_agg = (
    df.groupby("xyz_campaign_id")
    .agg(
        total_spent       = ("spent",               "sum"),
        total_revenue     = ("revenue",             "sum"),
        total_impressions = ("impressions",         "sum"),
        total_clicks      = ("clicks",              "sum"),
        total_conversoes  = ("approved_conversion", "sum"),
        roi_medio         = ("roi",                 "mean"),
        conv_rate_media   = ("conversion_rate",     "mean"),
        cpa_medio         = ("cpa",                 "mean"),
        n_anuncios        = ("ad_id",               "count"),
    )
    .reset_index()
)

# ROI calculado sobre os totais (mais preciso que média de médias)
mart_agg["roi_agregado"] = (
    (mart_agg["total_revenue"] - mart_agg["total_spent"]) / mart_agg["total_spent"]
)

out_agg = PROCESSED / "mart_aggregated.csv"
mart_agg.to_csv(out_agg, index=False)

print(f"mart_aggregated → {len(mart_agg)} linhas | {out_agg}")
print(mart_agg)

mart_aggregated → 3 linhas | ..\data\processed\mart_aggregated.csv
   xyz_campaign_id   total_spent  total_revenue  total_impressions  \
0              916    149.710001          800.0             448046   
1              936   2893.369999         5900.0            7797942   
2             1178  55662.149959        43450.0          204699959   

   total_clicks  total_conversoes  roi_medio  conv_rate_media  cpa_medio  \
0           113                16   8.714313         0.257143   4.712500   
1          1984               118   8.145738         0.212104   8.440565   
2         36068               869   0.637446         0.052148  58.862604   

   n_anuncios  roi_agregado  
0          35      4.343664  
1         288      1.039145  
2         613     -0.219398  


## 4. Mart 3 — Parâmetros globais

Valores padrão que o Streamlit usa ao inicializar. O usuário pode sobrescrever `avg_order_value` via slider.

In [14]:
taxa_conv_global = df["approved_conversion"].sum() / df["clicks"].sum()
cpa_global       = df["spent"].sum() / df["approved_conversion"].sum()

mart_params = pd.DataFrame({
    "avg_order_value":    [AVG_ORDER_VALUE],
    "taxa_conv_global":   [round(taxa_conv_global, 6)],
    "cpa_global":         [round(cpa_global, 4)],
    "total_anuncios":     [len(df)],
    "total_spent_global": [round(df["spent"].sum(), 2)],
})

out_params = PROCESSED / "mart_roi_parameters.csv"
mart_params.to_csv(out_params, index=False)

print(f"mart_roi_parameters → {len(mart_params)} linha | {out_params}")
print(mart_params.T)

mart_roi_parameters → 1 linha | ..\data\processed\mart_roi_parameters.csv
                               0
avg_order_value        50.000000
taxa_conv_global        0.026281
cpa_global             58.529600
total_anuncios        936.000000
total_spent_global  58705.230000


## 5. Verificação dos três marts

In [15]:
for nome, caminho in [
    ("mart_ads_performance",  out_ads),
    ("mart_aggregated",       out_agg),
    ("mart_roi_parameters",   out_params),
]:
    tmp = pd.read_csv(caminho)
    kb  = caminho.stat().st_size / 1024
    print(f"{nome:30s} | {len(tmp):>5} linhas | {tmp.shape[1]:>2} colunas | {kb:.1f} KB")

mart_ads_performance           |   936 linhas | 14 colunas | 77.2 KB
mart_aggregated                |     3 linhas | 11 colunas | 0.5 KB
mart_roi_parameters            |     1 linhas |  5 colunas | 0.1 KB


## 6. Esquema de cada mart

Tipos finais para referência ao implementar os módulos Python do Streamlit.

In [16]:
print("=== mart_ads_performance ===")
print(pd.read_csv(out_ads).dtypes)

print("\n=== mart_aggregated ===")
print(pd.read_csv(out_agg).dtypes)

print("\n=== mart_roi_parameters ===")
print(pd.read_csv(out_params).dtypes)

=== mart_ads_performance ===
ad_id                    int64
xyz_campaign_id          int64
age                        str
gender                     str
interest                 int64
impressions              int64
clicks                   int64
spent                  float64
total_conversion         int64
approved_conversion      int64
revenue                float64
roi                    float64
conversion_rate        float64
cpa                    float64
dtype: object

=== mart_aggregated ===
xyz_campaign_id        int64
total_spent          float64
total_revenue        float64
total_impressions      int64
total_clicks           int64
total_conversoes       int64
roi_medio            float64
conv_rate_media      float64
cpa_medio            float64
n_anuncios             int64
roi_agregado         float64
dtype: object

=== mart_roi_parameters ===
avg_order_value       float64
taxa_conv_global      float64
cpa_global            float64
total_anuncios          int64
total_spent_glob